In [0]:
# nb_01_bronze_ebird_ingest.py
#
# Widgets (set in Databricks Workflow or manually):
#   run_date  : YYYY-MM-DD  (default: yesterday)
#   run_id    : string      (default: auto-generated, used for lineage)
#
# What this notebook does:
#   1. Calls /product/lists/US-CA for run_date — paginated — to get all subIds
#   2. For each subId, calls /product/checklist/view/{subId} for full detail
#   3. Appends raw results to two Bronze Delta tables, partitioned by _pull_date
#   4. Errors on individual checklist fetches are recorded, not raised —
#      the job continues and failed subIds can be retried

import requests
import json
import uuid
import time
from datetime import datetime, date, timedelta
from typing import Iterator
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ── Config ────────────────────────────────────────────────────────────────────
EBIRD_TOKEN        = dbutils.secrets.get(scope="ebird", key="api_key")
REGION             = "US-CA"
CATALOG            = "birds"
DB                 = "ebird_bronze"
INDEX_TABLE        = f"{CATALOG}.{DB}.checklist_index"
DETAIL_TABLE       = f"{CATALOG}.{DB}.checklist_detail"
INDEX_PATH         = "/mnt/datalake/ebird/bronze/checklist_index"
DETAIL_PATH        = "/mnt/datalake/ebird/bronze/checklist_detail"

PAGE_SIZE          = 200    # max the API allows
THROTTLE_SECONDS   = 0.5    # sleep between detail calls — ~2 req/sec, well under limits
MAX_RETRIES        = 3      # per-request retry attempts
RETRY_BACKOFF      = 2.0    # seconds, doubles each retry

# ── Resolve run parameters ────────────────────────────────────────────────────
def get_widget(name: str, default: str) -> str:
    try:
        val = dbutils.widgets.get(name)
        return val if val.strip() else default
    except Exception:
        return default

yesterday   = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")
RUN_DATE    = get_widget("run_date", yesterday)
RUN_ID      = get_widget("run_id",   str(uuid.uuid4()))
PULL_DATE   = RUN_DATE   # partition key — the date we're pulling data FOR
INGEST_TS   = datetime.utcnow().isoformat()

print(f"run_date={RUN_DATE} | run_id={RUN_ID} | pull_date={PULL_DATE}")

# ── HTTP helper with retry ────────────────────────────────────────────────────
def get_with_retry(url: str, params: dict = None) -> requests.Response:
    headers = {"X-eBirdApiToken": EBIRD_TOKEN}
    backoff = RETRY_BACKOFF
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=headers, params=params, timeout=30)
            if resp.status_code == 429:
                # Rate limited — back off hard
                wait = int(resp.headers.get("Retry-After", 60))
                print(f"Rate limited. Waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp
        except requests.exceptions.RequestException as e:
            if attempt == MAX_RETRIES:
                raise
            print(f"Attempt {attempt} failed: {e}. Retrying in {backoff}s...")
            time.sleep(backoff)
            backoff *= 2

# ── Step 1: Fetch checklist index (paginated) ─────────────────────────────────
# /product/lists/{regionCode} returns checklists submitted for a region,
# sorted by submission date descending. We page through until we've collected
# all checklists whose obsDate matches our run_date.

def fetch_checklist_index(region: str, target_date: str) -> list[dict]:
    """
    Pages through /product/lists until obsDate no longer matches target_date.
    Returns flat list of raw checklist summary dicts, each annotated with
    _page_num for lineage.
    """
    url        = f"https://api.ebird.org/v2/product/lists/{region}"
    all_rows   = []
    page       = 1
    max_pages  = 2  # safety ceiling: 50 × 200 = 10,000 checklists

    while page <= max_pages:
        params = {
            "maxResults": PAGE_SIZE,
            "date":       target_date,  # YYYY-MM-DD — filters to this date only
        }
        resp      = get_with_retry(url, params)
        page_data = resp.json()

        if not page_data:
            break  # no more results

        for row in page_data:
            row["_page_num"] = page
        all_rows.extend(page_data)

        # API returns up to maxResults per call. If we got a full page,
        # there may be more — increment offset via the last subId.
        # If partial page, we're done.
        if len(page_data) < PAGE_SIZE:
            break

        page += 1
        time.sleep(0.25)  # light throttle between pagination calls

    print(f"Index: fetched {len(all_rows)} checklist stubs across {page} page(s)")
    return all_rows

index_rows = fetch_checklist_index(REGION, RUN_DATE)

if not index_rows:
    print(f"No checklists found for {RUN_DATE}. Exiting.")
    dbutils.notebook.exit("no_data")

# ── Write Bronze index table ──────────────────────────────────────────────────
# Same pattern as detail: sub_id + raw JSON + ingestion metadata only.
# Field extraction and nested parsing (loc.name etc.) belong in Silver.

index_schema = StructType([
    StructField("sub_id",        StringType(), False),
    StructField("raw_json",      StringType(), False),
    StructField("_pull_date",    StringType(), False),
    StructField("_ingestion_ts", StringType(), False),
    StructField("_run_id",       StringType(), False),
    StructField("_page_num",     IntegerType(), True),
])

index_records = [
    {
        "sub_id":        r.get("subId"),
        "raw_json":      json.dumps(r),
        "_pull_date":    PULL_DATE,
        "_ingestion_ts": INGEST_TS,
        "_run_id":       RUN_ID,
        "_page_num":     r.get("_page_num"),
    }
    for r in index_rows
]

index_df = spark.createDataFrame(index_records, schema=index_schema)

index_df.write.format("delta") \
    .mode("append") \
    .partitionBy("_pull_date") \
    .option("mergeSchema", "true") \
    .saveAsTable(INDEX_TABLE)

print(f"Wrote {len(index_records)} rows to {INDEX_TABLE}")

# ── Step 2: Fetch full checklist detail for each subId ────────────────────────
# One API call per subId. We record HTTP status and any error message so
# failures don't abort the run — they land in Bronze as error rows and
# can be retried by a separate notebook.

sub_ids = [r.get("subId") for r in index_rows if r.get("subId")]
print(f"Fetching detail for {len(sub_ids)} checklists...")

detail_records = []

for i, sub_id in enumerate(sub_ids):
    url           = f"https://api.ebird.org/v2/product/checklist/view/{sub_id}"
    http_status   = None
    fetch_error   = None
    raw_payload   = None

    try:
        resp        = get_with_retry(url)
        http_status = resp.status_code
        raw_payload = resp.text   # store raw — do not parse in Bronze
    except requests.exceptions.HTTPError as e:
        http_status = e.response.status_code if e.response else None
        fetch_error = str(e)
        print(f"  HTTP error for {sub_id}: {fetch_error}")
    except Exception as e:
        fetch_error = str(e)
        print(f"  Error for {sub_id}: {fetch_error}")

    detail_records.append({
        "sub_id":        sub_id,
        "raw_json":      raw_payload,   # None if fetch failed
        "_pull_date":    PULL_DATE,
        "_ingestion_ts": INGEST_TS,
        "_run_id":       RUN_ID,
        "_http_status":  http_status,
        "_fetch_error":  fetch_error,
    })

    # Throttle — 0.5s between calls = ~2 req/sec
    # For 2000 checklists this takes ~17 minutes, well within a daily window
    time.sleep(THROTTLE_SECONDS)

    if (i + 1) % 100 == 0:
        print(f"  {i + 1}/{len(sub_ids)} fetched...")

# ── Write Bronze detail table ─────────────────────────────────────────────────
detail_schema = StructType([
    StructField("sub_id",        StringType(), False),
    StructField("raw_json",      StringType(), True),   # nullable — error rows have None
    StructField("_pull_date",    StringType(), False),
    StructField("_ingestion_ts", StringType(), False),
    StructField("_run_id",       StringType(), False),
    StructField("_http_status",  IntegerType(), True),
    StructField("_fetch_error",  StringType(), True),
])

detail_df = spark.createDataFrame(detail_records, schema=detail_schema)

detail_df.write.format("delta") \
    .mode("append") \
    .partitionBy("_pull_date") \
    .option("mergeSchema", "true") \
    .saveAsTable(DETAIL_TABLE)

# ── Summary ───────────────────────────────────────────────────────────────────
success_count = sum(1 for r in detail_records if r["_fetch_error"] is None)
error_count   = sum(1 for r in detail_records if r["_fetch_error"] is not None)

print(f"""
Bronze ingest complete
──────────────────────────────
run_date      : {RUN_DATE}
run_id        : {RUN_ID}
checklists    : {len(sub_ids)}
detail ok     : {success_count}
detail errors : {error_count}
──────────────────────────────
""")

# Fail the job if more than 5% of detail calls errored
# — likely an API problem worth alerting on
error_rate = error_count / len(sub_ids) if sub_ids else 0
if error_rate > 0.05:
    raise Exception(
        f"Detail fetch error rate {error_rate:.1%} exceeds 5% threshold. "
        f"Check Bronze detail table for _run_id={RUN_ID}."
    )